# Action Catalog and Intents Viewer

**Schema(s)** defining this data:
- `contracts/schemas/action-catalog.schema.json` — Action catalog (actionType, group, requiresTarget, requiresRoomFeature, requiresEncounter).
- `contracts/schemas/action-intents.schema.json` — Intents (how actions map to intent).
- `contracts/schemas/action-policies.schema.json` — Policies (who can use which actions).
- `contracts/schemas/action-formulas.schema.json` — Formulas (e.g. narrative stat deltas). **Room feature** = room type only.

**Assets / packs** we load:
- `contracts/data/config_action_catalog.json` — action catalog
- `contracts/data/config_action_intents.json` — intents
- `contracts/data/config_action_policies.json` — policies
- `contracts/data/config_action_formulas.json` — formulas

**What this tool does:** View actions by group or intent; see requiresRoomFeature (room type) and other gating; rebalance formula deltas; behaviour = which action is available in which context (room feature, encounter).

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "packages" / "engine").is_dir() and (ROOT.parent / "packages" / "engine").is_dir():
    ROOT = ROOT.parent
CONTRACTS_DIR = ROOT / "packages" / "engine" / "src" / "escape-the-dungeon" / "contracts"
DATA = CONTRACTS_DIR / "data"

with open(DATA / "config_action_catalog.json", encoding="utf-8") as f:
    CATALOG = json.load(f)
with open(DATA / "config_action_intents.json", encoding="utf-8") as f:
    INTENTS = json.load(f)
with open(DATA / "config_action_policies.json", encoding="utf-8") as f:
    POLICIES = json.load(f)
with open(DATA / "config_action_formulas.json", encoding="utf-8") as f:
    FORMULAS = json.load(f)

print("Schemas: action-catalog, action-intents, action-policies, action-formulas")
print("Packs:   config_action_catalog, config_action_intents, config_action_policies, config_action_formulas")
print(f"Catalog: {len(CATALOG.get('actions', []))} actions")
print(f"Intents: {len(INTENTS.get('intents', []))}")
print(f"Policies: {len(POLICIES.get('policies', []))}")
print(f"Formulas keys: {list(FORMULAS.keys())}")

## View actions and gating

**Behaviour:** requiresRoomFeature (room type), requiresTarget, requiresEncounter. **Rebalance:** formula deltas in config_action_formulas.json.

In [ ]:
print("Action catalog (actionType, group, requiresRoomFeature, requiresTarget, requiresEncounter):")
for a in CATALOG.get("actions", []):
    room = a.get("requiresRoomFeature", "—")
    target = a.get("requiresTarget", False)
    enc = a.get("requiresEncounter", False)
    print(f"  {a.get('actionType', ''):20} group={a.get('group', ''):12} roomFeature={room} target={target} encounter={enc}")

print("\nActions gated by room feature (room type):")
for a in CATALOG.get("actions", []):
    if a.get("requiresRoomFeature"):
        print(f"  {a['actionType']} requiresRoomFeature={a['requiresRoomFeature']}")

print("\nFormulas (sample):")
for key in ["perFeatureCap", "actionDeltas"]:
    if key in FORMULAS:
        v = FORMULAS[key]
        if isinstance(v, dict) and len(v) > 3:
            print(f"  {key}: {list(v.keys())[:3]}...")
        else:
            print(f"  {key}: {v}")